# 5. Visualization & Early Warning System

This notebook produces final visualizations and evaluates the early warning system:
- Interactive Folium risk heatmap
- Ignition Risk Prioritization Curve
- Pre-ignition risk evolution timeline
- Early warning lead time analysis
- Dashboard data export

> **Prerequisite:** Run Notebooks 01-04 first.

## Daily Risk Aggregation

In [ ]:
# Use NDVI-enhanced predictions
test_df = test_df.copy()

# Ensure predictions exist
test_df["pred_prob"] = y_pred_proba_ndvi

# Compute daily average risk
daily_risk = (
    test_df.groupby("date")["pred_prob"]
    .mean()
    .reset_index()
    .sort_values("pred_prob", ascending=False)
)

daily_risk.head()

## Interactive Fire Risk Map (Folium)

In [ ]:
import folium
import numpy as np

# Select chosen day
selected_date = "2023-08-29"

day_df = test_df[test_df["date"] == selected_date].copy()

print("Grid cells that day:", len(day_df))
print("Ignitions that day:", day_df["ignition"].sum())

In [ ]:
lat_min = 32
lon_min = -125
grid_size = 0.25

day_df["lat"] = lat_min + (day_df["lat_bin"] + 0.5) * grid_size
day_df["lon"] = lon_min + (day_df["lon_bin"] + 0.5) * grid_size

day_df.head()

In [ ]:
# Center map roughly over California
m = folium.Map(location=[37, -120], zoom_start=6, tiles="cartodbpositron")

# Normalize probabilities for color scale
max_prob = day_df["pred_prob_ndvi"].max()
min_prob = day_df["pred_prob_ndvi"].min()

def get_color(prob):
    # simple red intensity scaling
    intensity = int(255 * (prob - min_prob) / (max_prob - min_prob + 1e-6))
    return f"#{intensity:02x}0000"

# Add grid cells
for _, row in day_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=get_color(row["pred_prob_ndvi"]),
        fill=True,
        fill_opacity=0.6,
        popup=f"Risk: {row['pred_prob_ndvi']:.3f}"
    ).add_to(m)

# Overlay ignition events
for _, row in day_df[day_df["ignition"] == 1].iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        popup="🔥 Ignition"
    ).add_to(m)

m

## Ignition Risk Prioritization Curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure sorted by predicted risk (descending)
test_sorted = test_df.sort_values("pred_prob_ndvi", ascending=False).reset_index(drop=True)

total_ignitions = test_sorted["ignition"].sum()
total_rows = len(test_sorted)

percentages = np.linspace(0.01, 1.0, 100)
capture_rates = []

for p in percentages:
    cutoff = int(p * total_rows)
    captured = test_sorted.iloc[:cutoff]["ignition"].sum()
    capture_rates.append(captured / total_ignitions)

capture_rates = np.array(capture_rates)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(percentages * 100, capture_rates * 100, label="Model", linewidth=2)

# Random baseline
plt.plot([0,100], [0,100], linestyle="--", label="Random Baseline")

plt.xlabel("Top % Highest-Risk Grid-Days Monitored")
plt.ylabel("% of Ignitions Captured")
plt.title("Ignition Risk Prioritization Curve (2023)")
plt.legend()
plt.grid(True)

plt.show()

The diagonal line represents random monitoring.
Our model’s curve rises sharply above it, meaning ignition risk is concentrated in a small fraction of grid-days.
For example, by monitoring just the top 20% highest-risk areas, we capture over 60% of ignition events.
This demonstrates that our system enables efficient pre-positioning of resources under limited operational capacity.

In [ ]:
# Pick one ignition event from that day
ignition_cells = day_df[day_df["ignition"] == 1]
ignition_cells.head()

## Pre-Ignition Risk Evolution

In [ ]:
# Recreate feature matrix for full dataset
features_with_ndvi = [
    "temp_7d_mean",
    "wind_7d_mean",
    "vpd_7d_mean",
    "precip_14d_sum",
    "elevation",
    "ndvi"
]

X_full = model_df[features_with_ndvi]
X_full_scaled = scaler.transform(X_full)

# Predict probabilities for entire dataset
model_df["pred_prob_ndvi"] = model_ndvi.predict_proba(X_full_scaled)[:, 1]

print("Predictions added to model_df.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Selected ignition cell
lat_sel = 20
lon_sel = 12

# Extract full history for that cell
cell_history = model_df[
    (model_df["lat_bin"] == lat_sel) &
    (model_df["lon_bin"] == lon_sel)
].copy()

# Sort by date
cell_history = cell_history.sort_values("date")

# Select last 14 days before ignition
ignition_date = pd.to_datetime("2023-08-29")

window = cell_history[
    (cell_history["date"] >= ignition_date - pd.Timedelta(days=14)) &
    (cell_history["date"] <= ignition_date)
].copy()

window = window.sort_values("date")

print(window[["date", "pred_prob_ndvi", "ignition"]])

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(window["date"], window["pred_prob_ndvi"], marker="o")

plt.axvline(ignition_date, color="red", linestyle="--", label="Ignition Day")

plt.title("Pre-Ignition Risk Evolution (Grid Cell 20,12)")
plt.ylabel("Predicted Ignition Probability")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Early Warning System Analysis

In [ ]:
# Compute daily 80th percentile threshold (top 20%)
daily_thresholds = (
    model_df[model_df["date"] >= "2023-01-01"]
    .groupby("date")["pred_prob_ndvi"]
    .quantile(0.80)
    .reset_index()
    .rename(columns={"pred_prob_ndvi": "top20_threshold"})
)

daily_thresholds.head()

In [ ]:
model_2023 = model_df[model_df["date"] >= "2023-01-01"].copy()

model_2023 = model_2023.merge(
    daily_thresholds,
    on="date",
    how="left"
)

# Flag whether cell is in top 20% on that day
model_2023["in_top20"] = (
    model_2023["pred_prob_ndvi"] >= model_2023["top20_threshold"]
).astype(int)

model_2023.head()

In [ ]:
import numpy as np

lead_times = []

ignitions_2023 = model_2023[model_2023["ignition"] == 1]

for _, row in ignitions_2023.iterrows():
    lat = row["lat_bin"]
    lon = row["lon_bin"]
    ignition_date = row["date"]

    # Look back 7 days
    history = model_2023[
        (model_2023["lat_bin"] == lat) &
        (model_2023["lon_bin"] == lon) &
        (model_2023["date"] < ignition_date) &
        (model_2023["date"] >= ignition_date - pd.Timedelta(days=7))
    ].sort_values("date")

    # Find earliest day in top 20%
    top_days = history[history["in_top20"] == 1]

    if not top_days.empty:
        first_warning_day = top_days.iloc[0]["date"]
        lead_time = (ignition_date - first_warning_day).days
        lead_times.append(lead_time)

lead_times = np.array(lead_times)

print("Number of ignition events with early warning:", len(lead_times))
print("Average lead time:", lead_times.mean())
print("Median lead time:", np.median(lead_times))

In [ ]:
coverage = len(lead_times) / len(ignitions_2023)
print("Early warning coverage:", coverage)

## Lead Time Distribution

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(lead_times, bins=range(0,9), align='left', rwidth=0.8)

plt.xticks(range(0,8))
plt.xlabel("Lead Time (Days Before Ignition)")
plt.ylabel("Number of Ignition Events")
plt.title("Early Warning Lead Time Distribution (2023)")
plt.grid(axis='y')

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.text(0.5, 0.7, f"Early Warning Coverage: {coverage*100:.1f}%",
         fontsize=16, ha='center')

plt.text(0.5, 0.5, f"Average Lead Time: {lead_times.mean():.1f} days",
         fontsize=14, ha='center')

plt.text(0.5, 0.35, f"Median Lead Time: {np.median(lead_times):.0f} days",
         fontsize=14, ha='center')

plt.axis("off")
plt.title("Early Warning Performance Summary (2023)")

plt.show()

## Dashboard Data Export

In [ ]:
print(full_dataset.columns)

In [ ]:
print(model_df.columns)

In [ ]:
model_df["date"] = pd.to_datetime(model_df["date"])

df_2023 = model_df[model_df["date"].dt.year == 2023].copy()

print(df_2023.shape)

In [ ]:
lat_min = 32
lon_min = -125
grid_size = 0.25

df_2023["lat"] = lat_min + (df_2023["lat_bin"] + 0.5) * grid_size
df_2023["lon"] = lon_min + (df_2023["lon_bin"] + 0.5) * grid_size

In [ ]:
dashboard_df = df_2023[[
    "date",
    "lat",
    "lon",
    "pred_prob_ndvi",
    "ignition"
]].copy()

dashboard_df.to_csv("dashboard_data_2023.csv", index=False)

print("Dashboard dataset shape:", dashboard_df.shape)
dashboard_df.head()